# Data Generation
Generates synthetic wealth analytics data: users, sessions, events, trades.
Outputs CSVs to GCS `/raw/` layer.

In [9]:
# Cell 1 — Auth (every child notebook has this)
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated")

✅ Authenticated


In [10]:
# ── Install dependencies ──────────────────────────────────
!pip install faker pandas -q


In [11]:
# ── Imports ───────────────────────────────────────────────
import pandas as pd
import random
import uuid
from datetime import timedelta
from faker import Faker

fake = Faker()
NUM_USERS = 1000
print('✅ Imports ready')


✅ Imports ready


In [12]:
# ── 1. Users ──────────────────────────────────────────────
users = []
for i in range(NUM_USERS):
    users.append({
        'user_id': i + 1,
        'signup_date': fake.date_between(start_date='-1y', end_date='today'),
        'country': random.choice(['India', 'USA', 'UK']),
        'account_type': random.choice(['retail', 'premium']),
        'risk_profile': random.choice(['low', 'medium', 'high'])
    })
users_df = pd.DataFrame(users)
print(f'✅ Users: {len(users_df)} rows')
users_df.head()


✅ Users: 1000 rows


,user_id,signup_date,country,account_type,risk_profile
0,1,2025-05-16,UK,premium,low
1,2,2026-03-23,USA,retail,medium
2,3,2026-02-04,USA,retail,medium
3,4,2025-09-08,India,premium,medium
4,5,2025-09-28,India,premium,high


In [13]:
# ── 2. Sessions ───────────────────────────────────────────
sessions = []
for i in range(3000):
    user_id = random.randint(1, NUM_USERS)
    start_time = fake.date_time_between(start_date='-30d', end_date='now')
    end_time = start_time + timedelta(minutes=random.randint(1, 120))
    sessions.append({
        'session_id': str(uuid.uuid4()),
        'user_id': user_id,
        'session_start': start_time,
        'session_end': end_time,
        'device': random.choice(['mobile', 'web'])
    })
sessions_df = pd.DataFrame(sessions)
print(f'✅ Sessions: {len(sessions_df)} rows')
sessions_df.head()


✅ Sessions: 3000 rows


,session_id,user_id,session_start,session_end,device
0,8902794a-1bfa-488c-bc84-3c0ca21e63b5,873,2026-03-06 04:07:41.328941,2026-03-06 04:47:41.328941,mobile
1,7154d56f-414f-4bd7-b098-08e28f0a427b,360,2026-03-11 08:21:54.937619,2026-03-11 09:15:54.937619,web
2,cc026aff-99d5-4b89-99f2-0cb05e3296e8,399,2026-03-05 07:12:00.359260,2026-03-05 07:46:00.359260,web
3,ed2bbbe9-8c10-42e6-be61-c87f214dda92,748,2026-03-22 07:22:21.029572,2026-03-22 08:07:21.029572,web
4,65c92d70-44b9-4d88-a9ae-e63d75f6acfd,471,2026-03-17 17:29:41.416078,2026-03-17 17:34:41.416078,mobile


In [14]:
# ── 3. Events ─────────────────────────────────────────────
event_types = ['login', 'view_stock', 'search_stock', 'add_watchlist', 'place_order', 'logout']
events = []
for i in range(10000):
    session = sessions_df.sample(1).iloc[0]
    event_time = session['session_start'] + timedelta(
        minutes=random.randint(0, int((session['session_end'] - session['session_start']).seconds / 60))
    )
    events.append({
        'event_id': str(uuid.uuid4()),
        'user_id': session['user_id'],
        'session_id': session['session_id'],
        'event_type': random.choice(event_types),
        'event_time': event_time
    })
events_df = pd.DataFrame(events)
print(f'✅ Events: {len(events_df)} rows')
events_df.head()


✅ Events: 10000 rows


,event_id,user_id,session_id,event_type,event_time
0,77274384-4f09-48f9-ac6c-79b0b181b150,510,bbc817ac-a98b-45d6-86ce-53f931e2def4,view_stock,2026-03-18 16:32:17.836724
1,a3e165ce-db50-4162-9475-7844f25011ca,469,b290f7f8-17d7-46ac-80de-a688ad6a2b15,view_stock,2026-03-24 01:26:19.815243
2,b3d61d32-e0d4-4116-86b2-d4127293206f,765,f5ef2132-1646-4070-abf3-5376885810de,login,2026-03-13 14:10:19.672479
3,c6c42232-5bea-4014-b6ac-1868100aa6d2,669,f48e7dc3-aac5-4b15-bf45-ed49f732b1d4,view_stock,2026-03-12 14:24:21.249099
4,093e23ae-be90-4cf2-8a4b-8f711bf21067,759,99d56c8c-40a6-4d30-80f6-85d50ead7cbf,logout,2026-03-17 07:30:07.836726


In [15]:
# ── 4. Trades ─────────────────────────────────────────────
stocks = ['AAPL','MSFT','GOOGL','AMZN','TSLA','NVDA','META','NFLX','INTC','AMD',
          'ADBE','CSCO','PEP','AVGO','COST','TMUS','QCOM','TXN','AMAT','INTU',
          'PYPL','SBUX','ISRG','BKNG','GILD','ADP','MDLZ','VRTX','LRCX','MU']

order_events = events_df[events_df['event_type'] == 'place_order']
trades = []
for i in range(len(order_events)):
    event = order_events.iloc[i]
    trades.append({
        'trade_id': str(uuid.uuid4()),
        'user_id': event['user_id'],
        'stock': random.choice(stocks),
        'trade_type': random.choice(['buy', 'sell']),
        'quantity': random.randint(1, 50),
        'price': round(random.uniform(50, 3000), 2),
        'trade_time': event['event_time']
    })
trades_df = pd.DataFrame(trades)
print(f'✅ Trades: {len(trades_df)} rows')
trades_df.head()


✅ Trades: 1662 rows


,trade_id,user_id,stock,trade_type,quantity,price,trade_time
0,5a84fba9-e842-4082-8549-6a45c6e01151,409,ADP,buy,48,2939.13,2026-03-24 17:17:17.535487
1,fbbeedfe-1432-45f5-88c8-2fadd0178258,145,TMUS,sell,45,1311.91,2026-03-05 10:07:30.314010
2,5ba669ef-3bbd-4255-8ebd-2e7ac5d23fce,615,PEP,buy,34,2974.80,2026-03-18 20:45:17.765997
3,fc950707-fb32-4f06-be93-a5f1d7463bf4,917,TXN,buy,6,77.85,2026-03-22 06:22:56.259046
4,bcd4f062-7db9-480b-bfb8-a46c1cc1d8fa,641,SBUX,buy,41,786.07,2026-03-13 21:58:06.928458


In [16]:
# ── 5. Save CSVs locally ──────────────────────────────────
import os
os.makedirs('/tmp/raw', exist_ok=True)

users_df.to_csv('/tmp/raw/users.csv', index=False)
sessions_df.to_csv('/tmp/raw/sessions.csv', index=False)
events_df.to_csv('/tmp/raw/events.csv', index=False)
trades_df.to_csv('/tmp/raw/trades.csv', index=False)

print('✅ All CSVs saved to /tmp/raw/')
print('   Next: run 02_ingestion/ingest_to_bronze.ipynb')


✅ All CSVs saved to /tmp/raw/
   Next: run 02_ingestion/ingest_to_bronze.ipynb


In [17]:
# ── 6. Upload raw CSVs to GCS ─────────────────────────────
# Run this only after GCP auth is done in main_pipeline.ipynb
from google.cloud import storage

BUCKET_NAME = 'wealth-analytics-data-lake'
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

for name in ['users', 'sessions', 'events', 'trades']:
    local = f'/tmp/raw/{name}.csv'
    blob = bucket.blob(f'raw/{name}.csv')
    blob.upload_from_filename(local)
    print(f'  ✅ raw/{name}.csv → GCS')

print('\n✅ Raw layer upload complete')


  ✅ raw/users.csv → GCS
  ✅ raw/sessions.csv → GCS
  ✅ raw/events.csv → GCS
  ✅ raw/trades.csv → GCS

✅ Raw layer upload complete
